# 04 - Joint Multi-Task Training (Classification + Detection + Segmentation)

Combines all three heads on **one shared encoder** and fine-tunes them together, alternating batches between the segmentation source (SIIM-ACR) and the detection source (RSNA) - exactly like the original MultiCheXNet training scheme (`MTL_generatot` in the Keras repo), reimplemented as `MTLJointLoader`.

We warm-start from the single-head checkpoints trained in notebooks 01-03 (optional but recommended - it converges much faster than starting from ImageNet alone).

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.siim_acr import SIIMACRDataset, build_siim_dataframe, get_default_train_augmentation as seg_aug, train_val_split as seg_split
from src.datasets.rsna import RSNADataset, load_rsna_dataframe, get_default_train_augmentation as det_aug, train_val_split as det_split, rsna_collate_fn
from src.datasets.mtl_dataset import MTLJointLoader
from src.utils.visualize import show_training_curves

print("device:", config.DEVICE)


## 1. Datasets (same paths as before)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


# BUG FIX: the segmentation source (seg_train_ds/seg_val_ds) has to be
# only_positive=True to avoid the segmenter collapsing to "predict nothing"
# (see notebook 01). But MTLJointLoader used to *also* draw the
# classifier's "class" label from that same source for half of all
# batches - and since only_positive means every one of those images is
# guaranteed class=1 (pneumothorax), the classifier could learn a spurious
# "this image is from the SIIM domain -> predict pneumothorax" shortcut
# instead of real features. A real run of this notebook showed exactly
# this failure: val_acc stuck at ~42-44% (barely above the 33% chance
# level for 3 classes) and getting worse every epoch while train metrics
# kept improving - classic shortcut/overfitting signature.
#
# Fix: build the SIIM dataframe once, *unfiltered*, and split train/val on
# it directly (so both streams below share the exact same image-level
# train/val partition - no leakage). seg_train_ds/seg_val_ds still only use
# the positive subset of that split (for segmentation loss), while
# cls_train_ds/cls_val_ds use the *full* split (both positive and negative
# images) purely to give the classifier real label diversity from the SIIM
# domain. MTLJointLoader's cls_dataset= now round-robins across all three
# sources instead of two.
full_seg_df = build_siim_dataframe(SEG_CSV_PATH, SEG_IMG_PATH, only_positive=False)
det_df = load_rsna_dataframe(DET_CSV_PATH)

full_train_ids, full_val_ids = seg_split(full_seg_df, val_fraction=0.15)
positive_ids = set(full_seg_df.loc[full_seg_df["EncodedPixels"].str.strip() != "-1",
                                    "ImageId"].unique())
seg_train_ids = [i for i in full_train_ids if i in positive_ids]
seg_val_ids   = [i for i in full_val_ids if i in positive_ids]
det_train_ids, det_val_ids = det_split(det_df, val_fraction=0.15)

seg_train_ds = SIIMACRDataset(full_seg_df, seg_train_ids, augmentation=seg_aug())
seg_val_ds   = SIIMACRDataset(full_seg_df, seg_val_ids)
cls_train_ds = SIIMACRDataset(full_seg_df, full_train_ids, augmentation=seg_aug())
cls_val_ds   = SIIMACRDataset(full_seg_df, full_val_ids)
det_train_ds = RSNADataset(det_df, det_train_ids, DET_IMG_PATH, augmentation=det_aug())
det_val_ds   = RSNADataset(det_df, det_val_ids, DET_IMG_PATH)

print(len(seg_train_ids), "positive-only seg train images,",
      len(full_train_ids), "full cls train images (both classes)")

BATCH_SIZE = 8   # joint batches are a bit heavier (3 losses); 8 is safe on a T4
# num_workers=2 for throughput. You may see a harmless
# "AssertionError: can only test a child process" during worker cleanup
# under Kaggle/Jupyter - cosmetic only, doesn't affect training.
train_loader = MTLJointLoader(seg_train_ds, det_train_ds, cls_dataset=cls_train_ds,
                              batch_size=BATCH_SIZE, num_workers=2,
                              det_collate_fn=rsna_collate_fn)
val_loader = MTLJointLoader(seg_val_ds, det_val_ds, cls_dataset=cls_val_ds,
                            batch_size=BATCH_SIZE, num_workers=2, shuffle=False,
                            det_collate_fn=rsna_collate_fn)
print(len(train_loader), "train batches/epoch,", len(val_loader), "val batches/epoch")


## 2. Build the full 3-headed model

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=True,
                     add_detector=True, add_segmenter=True).to(config.DEVICE)
print("Total params (M):", sum(p.numel() for p in model.parameters())/1e6)


## 3. (Optional but recommended) warm-start each head from its single-task checkpoint

Because every head + the shared encoder are just named `nn.Module`s, we can load each single-task checkpoint's `state_dict` and copy over only the matching keys (`strict=False`), instead of the manual layer-index surgery the original Keras code needed.

In [ ]:
def load_partial(model, ckpt_path):
    if not ckpt_path:
        return
    state = torch.load(ckpt_path, map_location=config.DEVICE)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded {ckpt_path}: {len(state)-len(missing)} tensors matched, "
          f"{len(missing)} missing, {len(unexpected)} unexpected")

load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt")
load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt")
load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/classifier_best.pt")


## 4. Joint fine-tuning

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)

# Same reasoning as notebook 01: give the scheduler (patience=2) room to
# drop the LR a couple of times and let early stopping (not a hardcoded
# epoch count) decide when training is actually done, rather than risking
# a cutoff mid-improvement.
EPOCHS = 30
LOSS_WEIGHTS = (1.0, 1.0, 1.0)   # (classification, detection, segmentation)

history = engine.fit(
    engine.train_one_epoch_mtl, engine.evaluate_mtl,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/mtl_best.pt",
    monitor="loss", mode="min", scaler=scaler, loss_weights=LOSS_WEIGHTS,
    # max_grad_norm=5.0 (engine default) - guards against the same kind of
    # mid-training divergence spike observed in notebook 01 without it.
    scheduler=scheduler, patience=7, pos_weight="auto",
)


## 5. Training curves

In [ ]:
show_training_curves({k: v for k, v in history.items() if k.endswith('loss')}, figsize=(20,4))
show_training_curves({k: v for k, v in history.items() if 'acc' in k})
show_training_curves({k: v for k, v in history.items() if 'pos_dice' in k})


## 6. Qualitative check on a validation image from each source

In [ ]:
from src.utils.visualize import show_segmentation_grid, show_detection_grid
from src.utils.bbox_utils import decode_predictions

model.load("/kaggle/working/multi-task-chest-xray-analysis-system/mtl_best.pt", map_location=config.DEVICE)
model.eval()

N_EXAMPLES = 8
CONF_THRESHOLD = 0.15  # see notebook 02 for why a lower/diagnostic threshold is used here

# segmentation grid (matches upstream README's "Segmentation results" style)
images, gt_masks, pred_masks = [], [], []
for i in range(min(N_EXAMPLES, len(seg_val_ds))):
    img, mask, label = seg_val_ds[i]
    with torch.no_grad():
        out = model(img.unsqueeze(0).to(config.DEVICE))
    images.append(img.permute(1, 2, 0).numpy())
    gt_masks.append(mask[0].numpy())
    pred_masks.append(out["seg"][0, 0].cpu().numpy())
show_segmentation_grid(images, gt_masks, pred_masks, n_cols=4)

# detection grid (matches upstream README's "Detection results" style)
positive_idxs = [i for i in range(len(det_val_ds)) if len(det_val_ds[i][3]) > 0][:N_EXAMPLES]
images, gt_boxes_list, pred_boxes_list, pred_scores_list = [], [], [], []
for idx in positive_idxs:
    img, target, label, gt_boxes = det_val_ds[idx]
    with torch.no_grad():
        out = model(img.unsqueeze(0).to(config.DEVICE))
    preds = decode_predictions(out["det"][0], conf_threshold=CONF_THRESHOLD)
    images.append(img.permute(1, 2, 0).numpy())
    gt_boxes_list.append(gt_boxes)
    pred_boxes_list.append([p["box"] for p in preds])
    pred_scores_list.append([p["score"] for p in preds])
show_detection_grid(images, gt_boxes_list, pred_boxes_list, pred_scores_list, n_cols=4)


Final joint checkpoint saved to `/content/mtl_best.pt`. This is the file you'll load in `05_Inference_and_Deployment.ipynb`.